In [2]:
import spacy

In [4]:
nlp = spacy.load('en_core_web_sm')

In [9]:
text = "Sam works at Google in San Francisco since 2023."

doc = nlp(text)

In [10]:
for i in doc.ents:
    print(f'{i.text} --- {i.label_}')

Sam --- PERSON
Google --- ORG
San Francisco --- GPE
2023 --- DATE


# Custom spacy Model

In [11]:
TRAIN_DATA = [
    ("Aswin joined Brototype in Kerala", 
     {"entities": [(0, 5, "PERSON"), (13, 22, "ORG"), (26, 32, "GPE")]}),
    
    ("He is studying Machine Learning at Brototype",
     {"entities": [(15, 31, "SKILL"), (35, 44, "ORG")]})
]


In [12]:
from spacy.training.example import Example

nlp = spacy.blank("en")

ner = nlp.add_pipe("ner")

ner.add_label("PERSON")
ner.add_label("ORG")
ner.add_label("GPE")
ner.add_label("SKILL")

nlp.initialize()


In [ ]:
import random

for epoch in range(20):
    random.shuffle(TRAIN_DATA)
    losses = {}
    
    for text, annotations in TRAIN_DATA:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], losses=losses)
    
    print(f"Epoch {epoch+1}, Losses: {losses}")






Epoch 1, Losses: {'ner': np.float32(11.275482)}
Epoch 2, Losses: {'ner': np.float32(10.889824)}
Epoch 3, Losses: {'ner': np.float32(9.911845)}
Epoch 4, Losses: {'ner': np.float32(8.181188)}
Epoch 5, Losses: {'ner': np.float32(6.032535)}
Epoch 6, Losses: {'ner': np.float32(4.3559256)}
Epoch 7, Losses: {'ner': np.float32(3.1307793)}
Epoch 8, Losses: {'ner': np.float32(6.734931)}
Epoch 9, Losses: {'ner': np.float32(3.0148616)}
Epoch 10, Losses: {'ner': np.float32(2.3646789)}
Epoch 11, Losses: {'ner': np.float32(1.534463)}
Epoch 12, Losses: {'ner': np.float32(1.072919)}
Epoch 13, Losses: {'ner': np.float32(0.8551564)}
Epoch 14, Losses: {'ner': np.float32(0.22225268)}
Epoch 15, Losses: {'ner': np.float32(0.040201675)}
Epoch 16, Losses: {'ner': np.float32(0.003919344)}
Epoch 17, Losses: {'ner': np.float32(0.00029267435)}
Epoch 18, Losses: {'ner': np.float32(7.850913e-05)}
Epoch 19, Losses: {'ner': np.float32(1.5816786e-06)}
Epoch 20, Losses: {'ner': np.float32(1.8805094e-06)}


In [15]:
test_text = "Aswin is learning Machine Learning in Brotype Kerala"

doc = nlp(test_text)

for ent in doc.ents:
    print(ent.text, ent.label_)


Aswin PERSON
Machine Learning SKILL
Brotype ORG
Kerala GPE


In [16]:
nlp.to_disk("custom_ner_model")

nlp = spacy.load("custom_ner_model")


# Vader

In [20]:
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk


sia = SentimentIntensityAnalyzer()



In [26]:
text = "This product is not amazing! Totally worth the money."

score = sia.polarity_scores(text)
print(score)


{'neg': 0.428, 'neu': 0.572, 'pos': 0.0, 'compound': -0.6424}


# TextBlob

In [28]:
from textblob import TextBlob

text = "The service was slow and disappointing."

blob = TextBlob(text)
print(blob.sentiment)


Sentiment(polarity=-0.45, subjectivity=0.5499999999999999)


# Transformers

In [2]:
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
text = ["I loved this product!", "This was a terrible experience"]

inputs = tokenizer(
    text,
    padding=True,
    truncation=True,
    return_tensors="pt"
)


In [4]:
outputs = model(**inputs)
logits = outputs.logits
print(logits)


tensor([[0.5179, 0.1769],
        [0.5584, 0.0424]], grad_fn=<AddmmBackward0>)


### Fine Tuning Bert

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


2025-12-18 12:20:28.907302: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-18 12:20:28.918467: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-18 12:20:29.396641: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-18 12:20:31.681289: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [1]:
from datasets import load_dataset
from transformers import Trainer, TrainingArguments


dataset = load_dataset('imdb')

2025-12-18 12:22:36.816341: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-18 12:22:36.828858: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-18 12:22:37.460364: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-18 12:22:40.863784: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [2]:
len(dataset)

3

In [3]:
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

In [4]:
dataset = dataset.map(tokenize, batched=True)
dataset = dataset.remove_columns(["text"])
dataset.set_format("torch")

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    
)

In [6]:
trainer = Trainer(model=model, args=training_args, train_dataset=dataset['train'])

In [ ]:
trainer.train()

/home/aswin/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


# GPT Pre-Trained

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")




In [ ]:
input_text = "The full form of AI is"
inputs = tokenizer.encode(input_text, return_tensors="pt")

outputs = model.generate(
    inputs,
    max_length=20,
    num_return_sequences=1,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)



The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [ ]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The full form of AI is not yet known, but it is likely to be a very interesting and


# LSTM

In [33]:
import numpy as np
import tensorflow as tf

text = open("shakespeare.txt", "r", encoding="utf-8").read().lower()

chars = sorted(list(set(text)))
char_to_idx = {c:i for i,c in enumerate(chars)}
idx_to_char = {i:c for c,i in char_to_idx.items()}

vocab_size = len(chars)




In [34]:
seq_length = 40
x_data = []
y_data = []

for i in range(0, len(text) - seq_length):
    seq = text[i:i+seq_length]
    target = text[i+seq_length]
    x_data.append([char_to_idx[c] for c in seq])
    y_data.append(char_to_idx[target])

X = np.array(x_data)
y = np.array(y_data)


In [35]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 64, input_length=seq_length),
    tf.keras.layers.LSTM(128),
    tf.keras.layers.Dense(vocab_size, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam"
)

model.summary()


/home/aswin/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2025-12-18 11:42:50.879920: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [36]:
model.fit(
    X, y,
    batch_size=128,
    epochs=5
)


Epoch 1/20
2213/2213 ━━━━━━━━━━━━━━━━━━━━ 271s 121ms/step - loss: 2.1873
Epoch 2/20
2213/2213 ━━━━━━━━━━━━━━━━━━━━ 230s 104ms/step - loss: 1.7775
Epoch 3/20
2213/2213 ━━━━━━━━━━━━━━━━━━━━ 237s 107ms/step - loss: 1.6400
Epoch 4/20
 773/2213 ━━━━━━━━━━━━━━━━━━━━ 2:09 90ms/step - loss: 1.5856

KeyboardInterrupt: 

In [44]:
def generate_text(start_text, length=15):
    input_seq = [char_to_idx[c] for c in start_text]
    result = start_text

    for _ in range(length):
        x = np.array(input_seq[-seq_length:]).reshape(1, -1)
        preds = model.predict(x, verbose=0)[0]
        next_idx = np.argmax(preds)
        next_char = idx_to_char[next_idx]


        result += next_char
        input_seq.append(next_idx)

    return result


In [45]:
print(generate_text("everyone must have"))


everyone must have stand,
and the


# Translation

In [ ]:
eng_sentences = [
    "i am a student",
    "i love machine learning",
    "how are you",
    "this is my book"
]

fra_sentences = [
    "<start> je suis un etudiant <end>",
    "<start> j aime l apprentissage automatique <end>",
    "<start> comment allez vous <end>",
    "<start> ceci est mon livre <end>"
]


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

eng_tokenizer = Tokenizer()
fra_tokenizer = Tokenizer()

eng_tokenizer.fit_on_texts(eng_sentences)
fra_tokenizer.fit_on_texts(fra_sentences)

eng_seq = eng_tokenizer.texts_to_sequences(eng_sentences)
fra_seq = fra_tokenizer.texts_to_sequences(fra_sentences)

max_eng_len = max(len(s) for s in eng_seq)
max_fra_len = max(len(s) for s in fra_seq)

encoder_input = pad_sequences(eng_seq, maxlen=max_eng_len, padding="post")
decoder_input = pad_sequences(fra_seq, maxlen=max_fra_len, padding="post")


In [ ]:
decoder_target = decoder_input[:, 1:]
decoder_input = decoder_input[:, :-1]


### Encoder

In [ ]:
from tensorflow.keras import layers, Model

eng_vocab_size = len(eng_tokenizer.word_index) + 1
fra_vocab_size = len(fra_tokenizer.word_index) + 1
embedding_dim = 128
latent_dim = 256

encoder_inputs = layers.Input(shape=(None,))
enc_emb = layers.Embedding(eng_vocab_size, embedding_dim)(encoder_inputs)
_, state_h, state_c = layers.LSTM(latent_dim, return_state=True)(enc_emb)

encoder_states = [state_h, state_c]


### Decoder

In [ ]:
decoder_inputs = layers.Input(shape=(None,))
dec_emb = layers.Embedding(fra_vocab_size, embedding_dim)(decoder_inputs)

decoder_lstm = layers.LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = layers.Dense(fra_vocab_size, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)


In [ ]:
tr_model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

tr_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

tr_model.summary()


In [ ]:
tr_model.fit(
    [encoder_input, decoder_input],
    decoder_target[..., None],
    epochs=30,
    verbose=0
)


In [ ]:
# Encoder inference model
encoder_model = tf.keras.Model(
    encoder_inputs,
    encoder_states
)


In [ ]:
# Decoder inference inputs
decoder_state_input_h = tf.keras.Input(shape=(latent_dim,))
decoder_state_input_c = tf.keras.Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Decoder embedding
dec_emb_inf = decoder_emb_layer(decoder_inputs)

# Decoder LSTM
decoder_outputs, state_h, state_c = decoder_lstm(
    dec_emb_inf,
    initial_state=decoder_states_inputs
)

decoder_states = [state_h, state_c]

# Dense layer
decoder_outputs = decoder_dense(decoder_outputs)

# Decoder inference model
decoder_model = tf.keras.Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)


In [ ]:
index_to_word = {i: w for w, i in fra_tokenizer.word_index.items()}
word_to_index = fra_tokenizer.word_index

start_token = word_to_index["<start>"]
end_token = word_to_index["<end>"]


In [ ]:
import numpy as np

def translate(sentence):
    # tokenize and pad input
    seq = eng_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_eng_len, padding="post")

    # encode
    states = encoder_model.predict(seq, verbose=0)

    # start token
    target_seq = np.array([[start_token]])

    translated_sentence = []

    for _ in range(max_fra_len):
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states,
            verbose=0
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = index_to_word.get(sampled_token_index, "")

        if sampled_word == "<end>":
            break

        translated_sentence.append(sampled_word)

        # update target sequence
        target_seq = np.array([[sampled_token_index]])

        # update states
        states = [h, c]

    return " ".join(translated_sentence)


In [ ]:
print(translate("i am a student"))


In [ ]:
print(translate("this is my book"))
